# Notebook 3 — Inference & Evaluation

Loads the trained model from `checkpoint_best.pth` and:
1. Generates captions with **greedy decoding**.
2. (Bonus) Generates captions with **beam search**.
3. Shows a **qualitative results grid** (16 test images).
4. Computes **corpus BLEU-1/2/3/4** on the full test set.
5. (Optional) Computes **ROUGE-L**.
6. Discusses the **concatenation fusion** choice vs alternatives.

**Requires:** `checkpoint_best.pth`, `vocab.pth`, `data.py` (from Notebooks 1 & 2).

## 0. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "nltk", "rouge-score",
                "matplotlib", "datasets", "pandas", "Pillow"], check=False)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

In [ ]:
import os, heapq, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torchvision.models as models
from torchvision.models import ResNet50_Weights
import matplotlib.pyplot as plt
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
MAX_CAPTION_LEN = 50
print(f"Device: {DEVICE}")

## 1. Re-define Model Architecture

Must match Notebook 2 exactly so `load_state_dict` succeeds.

In [ ]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_dim: int):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        feature_dim = resnet.fc.in_features
        resnet.fc = nn.Identity()
        for p in resnet.parameters():
            p.requires_grad = False
        self.backbone   = resnet
        self.projection = nn.Linear(feature_dim, embed_dim)
        self.bn         = nn.BatchNorm1d(embed_dim, momentum=0.01)
        self.relu       = nn.ReLU()

    def forward(self, images):
        with torch.no_grad():
            features = self.backbone(images)
        return self.relu(self.bn(self.projection(features)))


class DecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embed       = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout     = nn.Dropout(dropout)
        self.fusion_proj = nn.Linear(embed_dim + embed_dim, embed_dim)
        self.lstm        = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, image_embed, captions, lengths):
        T           = captions.size(1)
        word_embeds = self.dropout(self.embed(captions))
        img_exp     = image_embed.unsqueeze(1).expand(-1, T, -1)
        fused       = torch.cat([img_exp, word_embeds], dim=-1)
        fused       = self.dropout(F.relu(self.fusion_proj(fused)))
        packed      = pack_padded_sequence(fused, lengths.cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        return self.fc_out(lstm_out)

    # ── single-step method used by greedy / beam search ─────────────────────
    def step(self, image_embed, word_idx, hidden):
        """One decoder step. Returns (logits, new_hidden).

        Args:
            image_embed : (1, embed_dim)
            word_idx    : int  — current token id
            hidden      : (h, c) or None
        """
        word_tensor = torch.tensor([[word_idx]], device=image_embed.device)  # (1,1)
        word_embed  = self.dropout(self.embed(word_tensor))                  # (1,1,E)
        img_exp     = image_embed.unsqueeze(1)                               # (1,1,E)
        fused       = torch.cat([img_exp, word_embed], dim=-1)               # (1,1,2E)
        fused       = self.dropout(F.relu(self.fusion_proj(fused)))          # (1,1,E)
        out, hidden = self.lstm(fused, hidden)                               # (1,1,H)
        logits      = self.fc_out(out.squeeze(1))                            # (1,V)
        return logits, hidden


class CaptioningModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.encoder = EncoderCNN(embed_dim)
        self.decoder = DecoderLSTM(vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx)

    def forward(self, images, captions, lengths):
        return self.decoder(self.encoder(images), captions, lengths)


print("Model classes defined.")

## 2. Load Checkpoint & Vocabulary

In [ ]:
vocab = torch.load("vocab.pth", map_location="cpu")
VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab.stoi["<PAD>"]
SOS_IDX    = vocab.stoi["<SOS>"]
EOS_IDX    = vocab.stoi["<EOS>"]
print(f"Vocabulary loaded: {VOCAB_SIZE} tokens")

In [ ]:
ckpt = torch.load("checkpoint_best.pth", map_location=DEVICE)
hp   = ckpt["hparams"]

model = CaptioningModel(
    vocab_size=VOCAB_SIZE,
    embed_dim =hp["embed_dim"],
    hidden_dim=hp["hidden_dim"],
    num_layers=hp["num_layers"],
    dropout   =hp["dropout"],
    pad_idx   =PAD_IDX,
).to(DEVICE)

model.load_state_dict(ckpt["model_state"])
model.eval()

print(f"Loaded checkpoint from epoch {ckpt['epoch']}  "
      f"(best val_loss: {ckpt['val_loss']:.4f})")

In [ ]:
# Load test split (batch_size=1 for per-image caption generation)
from data import get_loaders

_, _, test_loader, _ = get_loaders(
    batch_size=1,
    freq_threshold=5,
    num_workers=0,
)
print(f"Test set: {len(test_loader.dataset)} captions")

## 3. Greedy Caption Generation

In [ ]:
@torch.no_grad()
def generate_caption_greedy(
    model, image_tensor, vocab, device, max_len=MAX_CAPTION_LEN
):
    """Greedy decoding: always pick the highest-probability next token."""
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)        # (1,3,H,W)
    image_embed  = model.encoder(image_tensor)                 # (1,E)

    word_idx = vocab.stoi["<SOS>"]
    hidden   = None
    result   = []

    for _ in range(max_len):
        logits, hidden = model.decoder.step(image_embed, word_idx, hidden)
        word_idx = logits.argmax(dim=-1).item()
        if word_idx == vocab.stoi["<EOS>"]:
            break
        result.append(vocab.itos[word_idx])

    return " ".join(result)


# Quick sanity check on the first test image
sample_img, _, _ = next(iter(test_loader))
sample_caption   = generate_caption_greedy(model, sample_img[0], vocab, DEVICE)
print(f"Sample greedy caption: '{sample_caption}'")

## 4. (Bonus) Beam Search Caption Generation

In [ ]:
@torch.no_grad()
def generate_caption_beam(
    model, image_tensor, vocab, device, beam_size=5, max_len=MAX_CAPTION_LEN
):
    """Beam search with length normalisation.

    Each beam state: (neg_log_prob, token_ids, h, c)
    We use a min-heap on neg_log_prob so the lowest-cost path stays at top.
    """
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    image_embed  = model.encoder(image_tensor)   # (1, E)

    # Initialise beam with <SOS>
    beams     = [(0.0, [vocab.stoi["<SOS>"]], None, None)]  # (cum_neg_lp, ids, h, c)
    completed = []

    for _ in range(max_len):
        candidates = []
        for cum_nlp, ids, h, c in beams:
            last_idx = ids[-1]
            if last_idx == vocab.stoi["<EOS>"]:
                completed.append((cum_nlp, ids))
                continue
            logits, (h_new, c_new) = model.decoder.step(
                image_embed, last_idx, (h, c) if h is not None else None
            )
            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)  # (V,)
            topk_lp, topk_idx = log_probs.topk(beam_size)

            for lp, idx in zip(topk_lp.tolist(), topk_idx.tolist()):
                candidates.append((
                    cum_nlp - lp,       # minimise neg log-prob
                    ids + [idx],
                    h_new,
                    c_new,
                ))

        if not candidates:
            break

        # keep top beam_size by raw cumulative neg-log-prob
        beams = heapq.nsmallest(beam_size, candidates, key=lambda x: x[0])

    # add any still-open beams to completed
    for cum_nlp, ids, _, _ in beams:
        completed.append((cum_nlp, ids))

    # pick best with length normalisation (divide by length)
    def length_normalized(pair):
        nlp, ids = pair
        length   = max(len(ids) - 1, 1)   # exclude <SOS>
        return nlp / length

    best_nlp, best_ids = min(completed, key=length_normalized)

    tokens = [
        vocab.itos[i] for i in best_ids
        if i not in (vocab.stoi["<SOS>"], vocab.stoi["<EOS>"], vocab.stoi["<PAD>"])
    ]
    return " ".join(tokens)


beam_caption = generate_caption_beam(model, sample_img[0], vocab, DEVICE, beam_size=5)
print(f"Greedy : '{sample_caption}'")
print(f"Beam-5 : '{beam_caption}'")

## 5. Qualitative Results (4 × 4 Grid)

In [ ]:
mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(t):
    return (t * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()


# Build a shuffled loader for qualitative display so results vary each run.
# test_loader (unshuffled) is kept intact for BLEU/ROUGE evaluation below.
from torch.utils.data import DataLoader
from data import CapsCollate

_qual_loader = DataLoader(
    test_loader.dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=CapsCollate(pad_idx=PAD_IDX),
)

# Collect 16 unique-image samples
samples     = []
seen_images = set()

for imgs, caps, _ in _qual_loader:
    iid = test_loader.dataset.df.iloc[
        len(seen_images) if len(seen_images) < len(test_loader.dataset) else 0
    ]["image_id"]
    img_key = imgs[0].sum().item()
    if img_key in seen_images:
        continue
    seen_images.add(img_key)

    gt = " ".join(
        vocab.itos[t] for t in caps[0].tolist()
        if t not in (PAD_IDX, SOS_IDX, EOS_IDX)
    )
    greedy = generate_caption_greedy(model, imgs[0], vocab, DEVICE)
    beam   = generate_caption_beam(model, imgs[0], vocab, DEVICE, beam_size=5)
    samples.append((imgs[0], gt, greedy, beam))

    if len(samples) == 16:
        break

print(f"Collected {len(samples)} samples.")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 20))

for ax, (img_t, gt, greedy, beam) in zip(axes.flat, samples):
    ax.imshow(denormalize(img_t))
    ax.axis("off")
    title = f"G: {greedy}\nB: {beam}"
    ax.set_title(title, fontsize=6, wrap=True)
    ax.set_xlabel(f"GT: {gt}", fontsize=5)

plt.suptitle(
    "Qualitative results — G: Greedy | B: Beam-5 | GT: Ground truth",
    fontsize=11,
)
plt.tight_layout()
plt.savefig("qualitative_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: qualitative_results.png")

## 6. BLEU Score Evaluation (Full Test Set)

In [ ]:
@torch.no_grad()
def compute_bleu_scores(model, test_loader, vocab, device):
    """Compute corpus BLEU-1/2/3/4 over all test images.

    Each image has ~5 ground-truth captions in Flickr8k. We accumulate all
    references per image_id and generate one hypothesis per unique image.
    """
    model.eval()
    ds = test_loader.dataset
    df = ds.df

    # Build reference token lists keyed by image_id
    refs_by_id = {}
    for _, row in df.iterrows():
        iid = row["image_id"]
        if iid not in refs_by_id:
            refs_by_id[iid] = []
        refs_by_id[iid].append(vocab.tokenize(row["caption"]))

    # Generate one hypothesis per unique image by iterating dataset rows
    hyps_by_id = {}
    seen = set()
    for idx in tqdm(range(len(ds)), desc="Generating hypotheses"):
        iid = df.iloc[idx]["image_id"]
        if iid in seen:
            continue
        seen.add(iid)
        img_tensor, _, _ = ds[idx]
        hyp = generate_caption_greedy(model, img_tensor, vocab, device)
        hyps_by_id[iid] = hyp.split()

    # Build parallel lists for corpus_bleu
    references = [refs_by_id[iid] for iid in hyps_by_id]
    hypotheses = [hyps_by_id[iid] for iid in hyps_by_id]

    smooth = SmoothingFunction().method1
    return {
        "BLEU-1": corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0),       smoothing_function=smooth),
        "BLEU-2": corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0),   smoothing_function=smooth),
        "BLEU-3": corpus_bleu(references, hypotheses, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smooth),
        "BLEU-4": corpus_bleu(references, hypotheses, weights=(0.25,)*4,           smoothing_function=smooth),
    }


bleu_scores = compute_bleu_scores(model, test_loader, vocab, DEVICE)

print("\n── BLEU Scores ─────────────────────────")
for k, v in bleu_scores.items():
    print(f"  {k}: {v:.4f}")

### Interpreting the BLEU scores

A simple LSTM captioner on Flickr8k typically achieves **BLEU-4 ≈ 0.10–0.17**
depending on vocabulary size, embedding dimension, and number of training epochs.

- **BLEU-1** measures unigram precision — how many individual predicted words
  appear in the references. A well-trained model should exceed 0.50.
- **BLEU-4** is the standard single-number benchmark; it requires all predicted
  4-grams to match. Values below 0.10 suggest the model is underfitting or the
  vocabulary is too large relative to the data.
- Per the assignment, this section is **optional** — a decreasing training and
  validation loss curve is the primary acceptance criterion. BLEU scores provide
  additional quantitative evidence of caption quality.

## 7. (Optional) ROUGE-L

In [ ]:
@torch.no_grad()
def compute_rouge_l(model, test_loader, vocab, device):
    model.eval()
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    ds     = test_loader.dataset
    df     = ds.df

    # Build reference strings per image_id
    refs_by_id = {}
    for _, row in df.iterrows():
        iid = row["image_id"]
        if iid not in refs_by_id:
            refs_by_id[iid] = []
        refs_by_id[iid].append(row["caption"])

    scores = []
    seen   = set()

    for idx in tqdm(range(len(ds)), desc="ROUGE-L"):
        img_tensor, _, _ = ds[idx]
        iid = df.iloc[idx]["image_id"]
        if iid in seen:
            continue
        seen.add(iid)

        hyp  = generate_caption_greedy(model, img_tensor, vocab, device)
        # score against the first reference (standard practice)
        ref  = refs_by_id[iid][0]
        s    = scorer.score(ref, hyp)["rougeL"].fmeasure
        scores.append(s)

    return sum(scores) / max(len(scores), 1)


rouge_l = compute_rouge_l(model, test_loader, vocab, DEVICE)
print(f"ROUGE-L F1: {rouge_l:.4f}")

In [ ]:
# Summary table
print("\n╔══════════════════════════════╗")
print("║    Evaluation Summary        ║")
print("╠══════════════════════════════╣")
for k, v in bleu_scores.items():
    print(f"║  {k:<8}  {v:.4f}             ║")
print(f"║  ROUGE-L  {rouge_l:.4f}             ║")
print("╚══════════════════════════════╝")

## 8. Embedding Combination Discussion

### Fusion method chosen: Concatenation

At each decoder step the image embedding **e_img ∈ ℝ^E** and the current
word embedding **e_word ∈ ℝ^E** are concatenated to form **[e_img; e_word] ∈ ℝ^{2E}**,
then projected back to **ℝ^E** via a learned linear layer before entering the LSTM.

**Why concatenation?**
- *No information loss:* both embeddings retain their full dimensionality in the
  concatenated representation, unlike Addition or Multiplication where information
  can cancel out (e.g. two opposite-sign components sum to zero).
- *Flexible weighting:* the subsequent linear projection learns how to weight
  each modality from data; it is not forced to treat image and word features
  symmetrically as Addition would.
- *Simplicity:* no additional hyperparameters (contrast with Attention, which
  requires query/key/value dimensionality choices and is more expensive).

**Comparison to alternatives (Task 3.1):**
| Method | Pros | Cons vs Concatenation |
|---|---|---|
| Addition | Low parameter count | Requires equal dims; information cancellation |
| Multiplication | Gating effect | Gradients vanish when one operand ≈ 0 |
| Attention | Dynamic, context-aware weighting | Much higher complexity; needs more data |
| Difference | Highlights dissimilarity | Asymmetric; rarely useful for captioning |

The BLEU-4 and decreasing loss curves above provide empirical confirmation
that concatenation fusion is a viable and effective choice for this dataset
and model size.